# Item Selection and Stopping Rules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/04_item_selection_and_stopping.ipynb)

After you understand the basic architecture, the next question is usually:
which CAT strategies should I start with? This notebook gives a compact
comparison of common initializer, selector, estimator, and stopper choices.

In [ ]:
import numpy as np

from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer, UniformInitializer
from catsim.item_bank import ItemBank
from catsim.selection import AStratSelector, LinearSelector, MaxInfoSelector, RandomSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import ConfidenceIntervalStopper, MaxLengthStopper, StandardErrorStopper, StopRule

In [ ]:
item_bank = ItemBank.generate_item_bank(160, seed=23)
examinees = np.linspace(-2.5, 2.5, 50)

## Strategy taxonomy

In a CAT loop:

- the initializer chooses the first ability estimate
- the selector chooses the next item
- the estimator updates ability after each response
- the stopper decides when the session ends

Different combinations change precision, test length, and exposure behavior.

## Compare a few selectors under a fixed-length CAT

In [ ]:
SEARCH_BOUND = 6.0  # NumericalSearchEstimator looks for theta in [-6, 6]
selector_configs = {
  "max_info": MaxInfoSelector(),
  "random": RandomSelector(),
  "linear": LinearSelector(list(range(15))),
  "a_strat": AStratSelector(test_size=15),
}

for name, selector in selector_configs.items():
  result = SimulationRunner(
    item_bank=item_bank,
    initializer=FixedPointInitializer(0.0),
    selector=selector,
    estimator=NumericalSearchEstimator(),
    stopper=MaxLengthStopper(15),
    seed=77,
  ).run(examinees)
  print({
    "selector": name,
    "rmse": round(float(result.rmse), 3),
    "bias": round(float(result.bias), 3),
    "estimates_at_bound": int(np.isclose(np.abs(result.latest_estimations), SEARCH_BOUND, atol=0.01).sum()),
    "overlap_rate": round(float(result.overlap_rate), 3),
  })

These numbers come from a single run (seed 77), so they are not a ranking:
with other seeds, the order of the four selectors changes. The `random` and
`linear` rows are also inflated by maximum-likelihood estimates stuck at the
edge of the search interval, -6 or 6 (`estimates_at_bound`).
`03_simulation_studies.ipynb` shows how to repeat a comparison over several
seeds.

## Compare stopping rules with the same selector

- `fixed_length` gives every examinee 15 items.
- `standard_error` stops once the standard error of the ability estimate is at
  most 0.4.
- `confidence_interval` is a classification rule. Its bounds are cut scores
  (here a single pass/fail cut at $\theta = 0$), and it stops once the 95%
  confidence interval around the estimate lies entirely on one side of the cut.

Each configuration is a `StopRule`, which ends a session as soon as any of its
stoppers holds. The two variable-length rules add a `MaxLengthStopper(40)` cap
and `min_items=6`, so they give at least 6 and at most 40 items. After the run,
`StopRule.satisfied` returns the stoppers that hold at the end of each session;
`stopped_at_max` counts the sessions in which the length cap is one of them
(for `fixed_length`, every session).

In [ ]:
stopper_configs = {
  "fixed_length": StopRule(MaxLengthStopper(15)),
  "standard_error": StopRule(StandardErrorStopper(0.4), MaxLengthStopper(40), min_items=6),
  "confidence_interval": StopRule(ConfidenceIntervalStopper([0.0], confidence=0.95), MaxLengthStopper(40), min_items=6),
}

for name, stopper in stopper_configs.items():
  result = SimulationRunner(
    item_bank=item_bank,
    initializer=UniformInitializer(),
    selector=MaxInfoSelector(),
    estimator=NumericalSearchEstimator(),
    stopper=stopper,
    seed=88,
  ).run(examinees)
  lengths = [session.administered_count for session in result.sessions]
  # the stoppers that hold when each session ends: the length cap, the target, or both
  held = [stopper.satisfied(item_bank, s.administered_item_ids, s.current_theta) for s in result.sessions]
  print({
    "stopper": name,
    "rmse": round(float(result.rmse), 3),
    "bias": round(float(result.bias), 3),
    "min_length": min(lengths),
    "mean_length": round(float(np.mean(lengths)), 1),
    "max_length": max(lengths),
    "stopped_at_max": sum(any(isinstance(h, MaxLengthStopper) for h in stoppers) for stoppers in held),
  })

The two variable-length rules adapt the test to the examinee. With
`standard_error`, most sessions stop once the target is met; the ones that run to
the cap belong to examinees away from the middle of the ability range, mostly
at its low end, where this bank has the fewest informative items. With
`confidence_interval`, examinees far from the cut score usually finish early,
while most of those close to it run to the cap. Precision is not its goal: it
stops once the pass/fail decision is settled, however imprecise the estimate.
Judge each rule by what it targets: precision, test length, or classification.

## Practical guidance

- `MaxInfoSelector` is a strong baseline when you want a classic adaptive CAT.
- `RandomSelector` and `LinearSelector` are useful baselines, not usually final production choices.
- Stratified selectors are useful when you want more structured item-bank usage.
- Fixed-length rules are simpler to compare.
- Precision-based rules such as `StandardErrorStopper` are often more realistic for adaptive delivery.

This notebook is meant to narrow the search space, not to replace a proper
methodological evaluation on your own item bank and population.